# Stage 3: TPIR Load Projection

**Purpose**: Filter Stage 2 data to Include-only rows and project to TPIR load schema

**Replicates Excel Workbook's "tpir_load" Sheet**:
- Filter to `include_flag = 'Include'`
- Project to 13-column TPIR schema
- Stub acquisition cost as 0.0 (PoC limitation)

**Input**: `individual_dfm_consolidated` (Stage 2 output)

**Output**: `tpir_load_equivalent`

**Row Count Expectation**: Subset of Stage 2 (e.g., 191 out of 220 for Brown Shipley template)

In [ ]:
# ========== Parameters ==========
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by orchestrator

try:
    period = mssparkutils.notebook.params.get("period") or period
    run_id = mssparkutils.notebook.params.get("run_id") or run_id
except Exception:
    pass

print(f"[STAGE 3] Starting TPIR load projection")
print(f"  Period: {period}")
print(f"  Run ID: {run_id}")

In [ ]:
# ========== Imports ==========
from pyspark.sql import functions as F

print("[STAGE 3] Imports complete")

## Step 1: Read Stage 2 Consolidated Data

Read `individual_dfm_consolidated` for the current period/run.

In [ ]:
# ========== Read Stage 2 Data ==========
print("\n[STEP 1] Reading Stage 2 consolidated data...")
print("=" * 70)

try:
    stage2_data = (
        spark.table("individual_dfm_consolidated")
        .filter((F.col("period") == period) & (F.col("run_id") == run_id))
    )
    stage2_count = stage2_data.count()
    print(f"  Stage 2 rows: {stage2_count}")
except Exception as e:
    print(f"[ERROR] Could not read individual_dfm_consolidated: {e}")
    print("  Run nb_stage2_brown_shipley first.")
    mssparkutils.notebook.exit("FAILED")

if stage2_count == 0:
    print("[ERROR] No Stage 2 data found for this period/run_id.")
    mssparkutils.notebook.exit("NO_DATA")

print("\n[STEP 1] Stage 2 data loaded successfully")

## Step 2: Filter to Include-Only Rows

Apply Excel workbook logic: only rows with `include_flag = 'Include'`.

In [ ]:
# ========== Filter to Include ==========
print("\n[STEP 2] Filtering to Include rows...")
print("=" * 70)

include_only = stage2_data.filter(F.col("include_flag") == "Include")
include_count = include_only.count()
removed_count = stage2_count - include_count

print(f"  Include rows: {include_count}")
print(f"  Removed rows: {removed_count}")
print(f"  Removal rate: {removed_count / stage2_count * 100:.1f}%")

if include_count == 0:
    print("[WARNING] All rows were excluded. Check exclusion logic.")
    mssparkutils.notebook.exit("NO_INCLUDE_ROWS")

print("\n[STEP 2] Filtering complete")

## Step 3: Project to TPIR Load Schema

Map to 13-column TPIR schema:
- Policyholder_Number
- Security_Code
- ISIN
- Other_Security_ID
- ID_Type
- Asset_Name
- Acq_Cost_in_GBP (stub: 0.0)
- Cash_Value_in_GBP
- Bid_Value_in_GBP
- Accrued_Interest
- Holding
- Loc_Bid_Price
- Currency_Local

In [ ]:
# ========== Project to TPIR Schema ==========
print("\n[STEP 3] Projecting to TPIR load schema...")
print("=" * 70)

tpir_load = include_only.select(
    # Provenance (for tracking, not in original TPIR schema)
    F.col("period"),
    F.col("run_id"),
    F.col("dfm_id"),
    F.col("source_row_id"),
    F.col("row_hash"),
    
    # TPIR schema columns
    F.col("policyholder_number").alias("Policyholder_Number"),
    F.col("security_code").alias("Security_Code"),
    F.col("isin").alias("ISIN"),
    F.col("other_security_id").alias("Other_Security_ID"),
    F.col("id_type").alias("ID_Type"),
    F.col("asset_name").alias("Asset_Name"),
    F.lit(0.0).cast("decimal(28,10)").alias("Acq_Cost_in_GBP"),  # Stub for PoC
    F.col("cash_value_gbp").cast("decimal(28,10)").alias("Cash_Value_in_GBP"),
    F.col("bid_value_gbp").cast("decimal(28,10)").alias("Bid_Value_in_GBP"),
    F.col("accrued_interest_gbp").cast("decimal(28,10)").alias("Accrued_Interest"),
    F.col("holding").cast("decimal(28,10)").alias("Holding"),
    F.col("local_bid_price").cast("decimal(28,10)").alias("Loc_Bid_Price"),
    F.col("local_currency").alias("Currency_Local"),
    
    # Metadata
    F.col("report_date"),
    F.current_timestamp().alias("projected_at")
)

tpir_count = tpir_load.count()
print(f"  TPIR load rows: {tpir_count}")

if tpir_count != include_count:
    print(f"  [WARNING] Row count changed during projection. Expected {include_count}, got {tpir_count}")

print("\n[STEP 3] Schema projection complete")

## Step 4: Write to tpir_load_equivalent

Persist Stage 3 output table.

In [ ]:
# ========== Write Stage 3 Output ==========
print("\n[STEP 4] Writing to tpir_load_equivalent table...")
print("=" * 70)

try:
    tpir_load.write.mode("append").saveAsTable("tpir_load_equivalent")
    print(f"  ✓ Successfully wrote {tpir_count} rows to tpir_load_equivalent")
except Exception as e:
    print(f"  [ERROR] Failed to write to tpir_load_equivalent: {e}")
    mssparkutils.notebook.exit("FAILED")

print("\n[STEP 4] Stage 3 output written successfully")

## Summary and Validation

Final statistics and reconciliation checks.

In [ ]:
# ========== Summary ==========
print("\n" + "=" * 70)
print("STAGE 3 TPIR PROJECTION COMPLETE")
print("=" * 70)

print(f"\nPeriod: {period}")
print(f"Run ID: {run_id}")

print(f"\nRow Counts:")
print(f"  Stage 2 input: {stage2_count}")
print(f"  Include rows: {include_count}")
print(f"  Removed rows: {removed_count}")
print(f"  TPIR load output: {tpir_count}")

print(f"\nValue Reconciliation:")
stage2_total = stage2_data.agg(F.sum("bid_value_gbp").alias("total")).collect()[0]["total"]
removed_total = stage2_data.filter(F.col("include_flag") == "Remove").agg(F.sum("bid_value_gbp").alias("total")).collect()[0]["total"]
tpir_total = tpir_load.agg(F.sum("Bid_Value_in_GBP").alias("total")).collect()[0]["total"]

print(f"  Stage 2 total bid value (GBP): £{stage2_total:,.2f}" if stage2_total else "  Stage 2 total: N/A")
print(f"  Removed total bid value (GBP): £{removed_total:,.2f}" if removed_total else "  Removed total: N/A")
print(f"  TPIR total bid value (GBP): £{tpir_total:,.2f}" if tpir_total else "  TPIR total: N/A")

if stage2_total and removed_total and tpir_total:
    expected_tpir = stage2_total - removed_total
    diff = abs(tpir_total - expected_tpir)
    print(f"  Expected TPIR total: £{expected_tpir:,.2f}")
    print(f"  Difference: £{diff:,.2f}")
    if diff < 0.01:  # Within 1 penny
        print("  ✓ Reconciliation PASSED")
    else:
        print(f"  ⚠ Reconciliation WARNING: difference of £{diff:,.2f}")

print(f"\nNext Steps:")
print(f"  1. Run nb_validate to generate dq_results and dq_exception_rows")
print(f"  2. Review validation results")
print(f"  3. Publish to downstream consumers")

print("\n" + "=" * 70)
mssparkutils.notebook.exit("OK")